# 🐱 KittyPau — ML/Analytics Notebook

**Propósito:** ejecutar un análisis reproducible sobre *toda la base* (scope=`all`) usando Supabase, con checks de calidad y fallback a tablas de analítica procesada.

## Qué produce este notebook
- `df_devices` (MAIN): inventario de dispositivos.
- `df_raw` (MAIN): lecturas en ventana temporal (`WINDOW_DAYS`).
- `df` (MAIN): derivadas por lectura (solo si readings trae métricas).
- `df_sessions` + `df_daily` (ANALYTICS): sesiones y resumen diario (recomendado si `readings` viene con métricas nulas).

## Orden de ejecución recomendado
1) Credenciales y conexión (carga `.env.notebook.local` y crea clientes)
2) Schema checks (debe dar OK en tablas/columnas)
3) Carga MAIN: `df_devices` → `df_raw` → Data availability checks
4) Derivadas MAIN: solo si hay métricas (`HAS_READING_METRICS=True`)
5) Carga ANALYTICS: `df_sessions` + `df_daily`
6) Baselines / EDA / Insights (elige automáticamente la mejor fuente)

## Seguridad
- Usa `Analisis_Estadistico_ML_IA/notebooks/.env.notebook.local` (ignorado por git).
- No pegues keys en celdas. Si ya expusiste una key, rótala en Supabase Dashboard.


## 0 · Instalación de dependencias

In [ ]:
# Ejecutar una sola vez en el entorno
# !pip install supabase python-dotenv pandas numpy matplotlib seaborn scipy scikit-learn plotly


## 1 ? Credenciales y conexión a Supabase

Este notebook carga variables desde:
- `Analisis_Estadistico_ML_IA/notebooks/.env.notebook.local`

### Variables mínimas (MAIN)
- `SUPABASE_MAIN_URL`
- `SUPABASE_MAIN_SERVICE_ROLE_KEY` (**sb_secret_*** o JWT legacy `eyJ...`)

### Variables opcionales (ANALYTICS)
- `SUPABASE_ANALYTICS_URL`
- `SUPABASE_ANALYTICS_SERVICE_ROLE_KEY`

### Parámetros de ejecución
- `ANALYSIS_SCOPE=all` (recomendado para análisis total)
- `WINDOW_DAYS`, `PAGE_SIZE`, `MAX_ROWS`

> Si ves `401 Invalid API key`: normalmente es porque **URL y KEY no son del mismo proyecto** o hay un typo al copiar la key.


In [ ]:
import os
from pathlib import Path
from supabase import create_client, Client

# =============================================================
# Env loading (Notebook-first)
# =============================================================
# Goal: load `Analisis_Estadistico_ML_IA/notebooks/.env.notebook.local`
# regardless of Jupyter's current working directory.

def load_notebook_env() -> Path:
    try:
        from dotenv import load_dotenv
    except Exception as e:
        raise RuntimeError(
            "Missing dependency: python-dotenv. Install with: pip install python-dotenv"
        ) from e

    cwd = Path.cwd().resolve()

    candidates = [
        cwd / ".env.notebook.local",
        cwd / "Analisis_Estadistico_ML_IA" / "notebooks" / ".env.notebook.local",
    ]

    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / "Analisis_Estadistico_ML_IA" / "notebooks" / ".env.notebook.local")

    env_path = next((c for c in candidates if c.exists()), None)
    if not env_path:
        raise FileNotFoundError(
            "Could not find `.env.notebook.local`. Create: Analisis_Estadistico_ML_IA/notebooks/.env.notebook.local"
        )

    load_dotenv(env_path, override=False)
    return env_path

ENV_PATH = load_notebook_env()
print(f"ENV loaded: {ENV_PATH}")

# =============================================================
# Notebook config
# =============================================================
ANALYSIS_SCOPE = (os.getenv("ANALYSIS_SCOPE", "all").strip().lower())  # all | pet
WINDOW_DAYS = int(os.getenv("WINDOW_DAYS", "30"))

PAGE_SIZE = int(os.getenv("PAGE_SIZE", "10000"))
MAX_ROWS = int(os.getenv("MAX_ROWS", "200000"))

# =============================================================
# Supabase MAIN (devices/readings)
# =============================================================
SUPABASE_MAIN_URL = (os.getenv("SUPABASE_MAIN_URL") or os.getenv("SUPABASE_URL") or "").strip()
SUPABASE_MAIN_SERVICE_ROLE_KEY = (
    os.getenv("SUPABASE_MAIN_SERVICE_ROLE_KEY")
    or os.getenv("SUPABASE_SERVICE_ROLE_KEY")
    or ""
).strip()

# =============================================================
# Supabase ANALYTICS (pet_sessions/pet_daily_summary) - optional
# =============================================================
SUPABASE_ANALYTICS_URL = (os.getenv("SUPABASE_ANALYTICS_URL") or "").strip()
SUPABASE_ANALYTICS_SERVICE_ROLE_KEY = (os.getenv("SUPABASE_ANALYTICS_SERVICE_ROLE_KEY") or "").strip()

TARGET_PET_ID = (os.getenv("PET_ID") or "").strip() if ANALYSIS_SCOPE == "pet" else None


def _require(name: str, value: str) -> str:
    if not value:
        raise ValueError(f"Missing env var: {name}")
    return value


def _looks_like_jwt(key: str) -> bool:
    return key.startswith("eyJ")


def _looks_like_secret(key: str) -> bool:
    return key.startswith("sb_secret_") or _looks_like_jwt(key)


def _validate_service_key(name: str, key: str) -> None:
    if not key:
        raise ValueError(f"Missing env var: {name}")
    if key.startswith("sb_publishable_"):
        raise ValueError(
            f"{name} is publishable (sb_publishable_*). For full-db analysis you must use a privileged key: sb_secret_* (or service_role JWT eyJ...)."
        )
    if not _looks_like_secret(key):
        print(
            f"WARNING: {name} does not look like sb_secret_* or a JWT. If you get 401 Invalid API key, re-copy the secret key from Supabase Dashboard."
        )


SUPABASE_MAIN_URL = _require("SUPABASE_MAIN_URL", SUPABASE_MAIN_URL)
_validate_service_key("SUPABASE_MAIN_SERVICE_ROLE_KEY", SUPABASE_MAIN_SERVICE_ROLE_KEY)

supabase_main: Client = create_client(SUPABASE_MAIN_URL, SUPABASE_MAIN_SERVICE_ROLE_KEY)

supabase_analytics: Client | None = None
if SUPABASE_ANALYTICS_URL or SUPABASE_ANALYTICS_SERVICE_ROLE_KEY:
    SUPABASE_ANALYTICS_URL = _require("SUPABASE_ANALYTICS_URL", SUPABASE_ANALYTICS_URL)
    _validate_service_key("SUPABASE_ANALYTICS_SERVICE_ROLE_KEY", SUPABASE_ANALYTICS_SERVICE_ROLE_KEY)
    supabase_analytics = create_client(SUPABASE_ANALYTICS_URL, SUPABASE_ANALYTICS_SERVICE_ROLE_KEY)

print("Supabase clients ready")
print(f"  scope: {ANALYSIS_SCOPE}")
print(f"  window_days: {WINDOW_DAYS}")
print(f"  main: {SUPABASE_MAIN_URL}")
print(f"  analytics: {'configured' if supabase_analytics else 'not configured'}")

# Quick pings (PostgREST)
_ = supabase_main.table("devices").select("id").limit(1).execute()
print("MAIN OK: access to devices")

if supabase_analytics is not None:
    _ = supabase_analytics.table("pet_sessions").select("id").limit(1).execute()
    print("ANALYTICS OK: access to pet_sessions")


In [ ]:
# =============================================================
# Schema checks (MAIN + ANALYTICS)
# =============================================================
# Goal: fail early if tables/columns are missing or if URL/key are mismatched.

from typing import Sequence

def check_table_columns(client: Client, table: str, columns: Sequence[str]) -> bool:
    cols = ",".join(columns)
    try:
        _ = client.table(table).select(cols).limit(1).execute()
        print(f"OK  {table} ({len(columns)} cols)")
        return True
    except Exception as e:
        msg = str(e)
        print(f"FAIL {table}")
        print(f"  columns: {cols}")
        print(f"  error: {msg[:240]}")
        return False

MAIN_TABLES = {
    "devices": [
        "id","device_id","device_type","status","device_state","plate_weight_grams",
        "battery_level","last_seen","pet_id","owner_id"
    ],
    "readings": [
        "id","device_id","pet_id","weight_grams","water_ml","flow_rate","temperature","humidity",
        "battery_level","recorded_at","ingested_at","clock_invalid"
    ],
}

print("")
print("MAIN schema checks")
main_ok = True
for t, cols in MAIN_TABLES.items():
    main_ok = check_table_columns(supabase_main, t, cols) and main_ok

analytics_ok = True
if supabase_analytics is not None:
    ANALYTICS_TABLES = {
        "pet_sessions": [
            "id","owner_id","pet_id","device_id","session_type","session_start","session_end",
            "duration_sec","grams_consumed","water_ml","classification","anomaly_score","baseline_grams",
            "avg_temperature","avg_humidity","is_premium_data","created_at"
        ],
        "pet_daily_summary": [
            "id","owner_id","pet_id","summary_date","total_food_grams","food_sessions","total_water_ml",
            "water_sessions","anomaly_count","skipped_meals","avg_temperature","avg_humidity","first_session_at",
            "last_session_at","processed_at","readings_processed"
        ],
    }

    print("")
    print("ANALYTICS schema checks")
    for t, cols in ANALYTICS_TABLES.items():
        analytics_ok = check_table_columns(supabase_analytics, t, cols) and analytics_ok
else:
    print("")
    print("Analytics client not configured; skipping analytics checks")

if not main_ok or not analytics_ok:
    raise RuntimeError(
        "Schema checks failed: revisa que las tablas/columnas existan y que URL+KEY sean del MISMO proyecto."
    )


## 2 · Helpers de carga y utilidades

In [ ]:
from supabase import Client
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, mannwhitneyu, ks_2samp
import warnings
warnings.filterwarnings('ignore')

# Estilo global de plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})


def supabase_to_df(
    table: str,
    client: Client | None = None,
    filters: dict | None = None,
    columns: str = '*',
    order_col: str | None = None,
    limit: int = 10_000,
    page_size: int | None = None,
) -> pd.DataFrame:
    """Carga una tabla de Supabase a DataFrame con paginación opcional.

    - Si `client` es None, usa `supabase_main`.
    - Para datasets grandes, usa `page_size` (o `PAGE_SIZE` del env) para paginar con `.range()`.
    """
    client = client or globals().get('supabase_main')
    if client is None:
        raise ValueError('supabase_to_df requiere client (o que exista supabase_main)')

    if not table:
        raise ValueError('supabase_to_df requiere `table`')

    if page_size is None:
        page_size = int(globals().get('PAGE_SIZE', 10_000))

    def build_query():
        q = client.table(table).select(columns)
        if filters:
            for col, val in filters.items():
                q = q.eq(col, val)
        if order_col:
            q = q.order(order_col, desc=False)
        return q

    if limit <= page_size:
        try:
            resp = build_query().limit(limit).execute()
        except Exception as e:
            raise RuntimeError(f"supabase_to_df failed: table={table}, columns={columns}") from e
        return pd.DataFrame(resp.data or [])

    rows: list[dict] = []
    offset = 0
    while len(rows) < limit:
        hi = min(offset + page_size - 1, offset + (limit - len(rows)) - 1)
        try:
            resp = build_query().range(offset, hi).execute()
        except Exception as e:
            raise RuntimeError(
                f"supabase_to_df paging failed: table={table}, offset={offset}, page_size={page_size}"
            ) from e
        batch = resp.data or []
        rows.extend(batch)
        if len(batch) < (hi - offset + 1):
            break
        offset += page_size

    return pd.DataFrame(rows)


def parse_timestamps(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Convierte columnas de timestamp a datetime UTC."""
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')
    return df


def effective_ts(df: pd.DataFrame) -> pd.Series:
    """Timestamp efectivo: clock_invalid=True ? ingested_at; si no ? recorded_at.

    Implementación vectorizada (evita apply fila-a-fila).
    """
    if 'clock_invalid' not in df.columns:
        return df['recorded_at']
    return pd.to_datetime(
        np.where(df['clock_invalid'].fillna(False), df['ingested_at'], df['recorded_at']),
        utc=True,
        errors='coerce',
    )


print('Helpers loaded')


## 3 ? Carga de datos

### 3.1 ? Dispositivos (MAIN)
Se carga `df_devices` desde `public.devices`.

### 3.2 ? Lecturas (MAIN)
Se carga `df_raw` desde `public.readings` en la ventana `WINDOW_DAYS`, con paginación para no cortar datasets grandes.

### 3.3 ? Anal?tica procesada (ANALYTICS)
Se cargan:
- `df_sessions` desde `public.pet_sessions`
- `df_daily` desde `public.pet_daily_summary`

**Nota importante:** si `readings` trae `weight_grams` y `water_ml` nulos (100%), la ?verdad analítica? estar? en `df_sessions/df_daily`.


In [ ]:
# ── Dispositivos ──────────────────────────────────────────────────────────────
df_devices = supabase_to_df(
    client=supabase_main,
    table="devices",
    columns="id, device_id, device_type, status, device_state, "
            "plate_weight_grams, battery_level, last_seen, pet_id, owner_id"
)
df_devices = parse_timestamps(df_devices, ['last_seen'])

print(f"📦 Dispositivos cargados: {len(df_devices)}")
df_devices


### 3.2 — Lecturas (últimos 30 días)

In [ ]:
from datetime import datetime, timezone, timedelta

PAGE_SIZE = int(os.getenv("PAGE_SIZE", "10000"))
MAX_ROWS = int(os.getenv("MAX_ROWS", "200000"))

since_dt = datetime.now(timezone.utc) - timedelta(days=WINDOW_DAYS)
since = since_dt.isoformat()

print(f"Window: last últimos {WINDOW_DAYS} días (desde {since_dt.strftime('%Y-%m-%d %H:%M')} UTC)")

READINGS_COLUMNS = (
    "id, device_id, pet_id, weight_grams, water_ml, flow_rate, "
    "temperature, humidity, battery_level, recorded_at, ingested_at, clock_invalid"
)

if ANALYSIS_SCOPE == "pet":
    if not TARGET_PET_ID:
        raise ValueError("ANALYSIS_SCOPE=pet requiere PET_ID")
    pet_devices = df_devices[df_devices['pet_id'] == TARGET_PET_ID]['id'].tolist()
    print(f"Devices for pet: {len(pet_devices)}")
else:
    pet_devices = []
    print("Scope=all: no pet_id filter (whole DB in window)")

all_readings: list[dict] = []

if ANALYSIS_SCOPE == "pet":
    for dev_id in pet_devices:
        offset = 0
        while len(all_readings) < MAX_ROWS:
            resp = (
                supabase_main.table("readings")
                .select(READINGS_COLUMNS)
                .eq("device_id", dev_id)
                .gte("ingested_at", since)
                .order("ingested_at", desc=False)
                .range(offset, offset + PAGE_SIZE - 1)
                .execute()
            )
            batch = resp.data or []
            all_readings.extend(batch)
            if len(batch) < PAGE_SIZE:
                break
            offset += PAGE_SIZE
else:
    offset = 0
    while len(all_readings) < MAX_ROWS:
        resp = (
            supabase_main.table("readings")
            .select(READINGS_COLUMNS)
            .gte("ingested_at", since)
            .order("ingested_at", desc=False)
            .range(offset, offset + PAGE_SIZE - 1)
            .execute()
        )
        batch = resp.data or []
        all_readings.extend(batch)
        if len(batch) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

print(f"Loaded readings: {len(all_readings):,} (cap MAX_ROWS={MAX_ROWS:,})")

df_raw = pd.DataFrame(all_readings)
if df_raw.empty:
    raise RuntimeError("No se cargaron lecturas. Revisa credenciales/ventana.")

df_raw = parse_timestamps(df_raw, ['recorded_at', 'ingested_at'])


In [ ]:
# =============================================================
# Data QA summary (MAIN)
# =============================================================
# Data quality quick checks to avoid running ML on broken inputs.

print('')
print('Data QA summary')

n_devices = len(df_devices) if 'df_devices' in globals() else None
print('devices:', n_devices)
print('readings:', len(df_raw))

# Effective timestamp range
_tmp = df_raw.copy()
_tmp['effective_ts'] = effective_ts(_tmp)

print('effective_ts min:', _tmp['effective_ts'].min())
print('effective_ts max:', _tmp['effective_ts'].max())

# clock_invalid rate
if 'clock_invalid' in _tmp.columns:
    rate = float(_tmp['clock_invalid'].fillna(False).mean())
    print('clock_invalid rate:', round(rate, 4))

# Duplicates by (device_id, recorded_at)
if 'device_id' in _tmp.columns and 'recorded_at' in _tmp.columns:
    dup = _tmp.duplicated(subset=['device_id','recorded_at']).sum()
    print('duplicate (device_id, recorded_at):', int(dup))

# Freshness per device (max effective_ts)
if 'device_id' in _tmp.columns:
    last = _tmp.groupby('device_id', dropna=False)['effective_ts'].max().reset_index()
    last = last.sort_values('effective_ts', ascending=True)
    print('devices with oldest last effective_ts:')
    display(last.head(10))


### 3.3 — Sesiones analíticas y resúmenes diarios

In [ ]:
# Sesiones + resumen diario (Supabase Analytics) — opcional
# Requiere SUPABASE_ANALYTICS_URL + SUPABASE_ANALYTICS_SERVICE_ROLE_KEY

from datetime import datetime, timezone, timedelta
from IPython.display import display

since_dt = datetime.now(timezone.utc) - timedelta(days=WINDOW_DAYS)
print(f"Analytics window: últimos {WINDOW_DAYS} días (desde {since_dt.strftime('%Y-%m-%d %H:%M')} UTC)")

# ------------------------------
# pet_sessions
# ------------------------------
df_sessions = pd.DataFrame()
if supabase_analytics is None:
    print("??  Analytics client no configurado: se omite carga de pet_sessions")
else:
    filters = {"pet_id": TARGET_PET_ID} if TARGET_PET_ID else None
    df_sessions = supabase_to_df(
        client=supabase_analytics,
        table="pet_sessions",
        filters=filters,
        columns=(
            "id, owner_id, pet_id, device_id, session_type, session_start, session_end, duration_sec, "
            "grams_consumed, water_ml, classification, anomaly_score, baseline_grams, avg_temperature, avg_humidity, created_at"
        ),
        order_col="session_start",
        limit=50_000,
    )
    df_sessions = parse_timestamps(df_sessions, ['session_start', 'session_end', 'created_at'])
    if 'session_start' in df_sessions.columns:
        df_sessions = df_sessions[df_sessions['session_start'] >= since_dt]
    print(f"?? Sesiones cargadas (ventana): {len(df_sessions):,}")

# ------------------------------
# pet_daily_summary
# ------------------------------
df_daily = pd.DataFrame()
if supabase_analytics is None:
    print("??  Analytics client no configurado: se omite carga de pet_daily_summary")
else:
    filters = {"pet_id": TARGET_PET_ID} if TARGET_PET_ID else None
    df_daily = supabase_to_df(
        client=supabase_analytics,
        table="pet_daily_summary",
        filters=filters,
        columns=(
            "id, owner_id, pet_id, summary_date, total_food_grams, food_sessions, total_water_ml, "
            "water_sessions, anomaly_count, skipped_meals, avg_temperature, avg_humidity, first_session_at, "
            "last_session_at, processed_at, readings_processed"
        ),
        order_col="summary_date",
        limit=10_000,
    )
    df_daily = parse_timestamps(df_daily, ['summary_date', 'first_session_at', 'last_session_at', 'processed_at'])
    if 'summary_date' in df_daily.columns:
        df_daily = df_daily[df_daily['summary_date'] >= since_dt]
    print(f"?? Daily summary cargado (ventana): {len(df_daily):,}")

print('')
print('Preview df_sessions:')
display(df_sessions.head(3))
print('')
print('Preview df_daily:')
display(df_daily.head(3))


## 4 ? Calidad de datos (Capa 1)

Objetivo: detectar temprano problemas antes de correr features/ML.

Incluye:
- Rango temporal real (`effective_ts min/max`)
- Missing-rate (not-null %) por variable
- `clock_invalid` rate
- Duplicados por `(device_id, recorded_at)`
- ?Freshness? por dispositivo (?ltimo `effective_ts`)

Si una variable aparece como **NO DATA** o 0% not-null, evita entrenar/alertar con esa señal y usa Analytics (sesiones/resumen diario).


In [ ]:
SENSOR_COLS = ['weight_grams', 'water_ml', 'flow_rate', 'temperature',
               'humidity', 'light_lux', 'light_percent', 'battery_level']

# Resumen de completitud
completeness = pd.DataFrame({
    'columna': SENSOR_COLS,
    'presentes': [df_raw[c].notna().sum() if c in df_raw.columns else 0 for c in SENSOR_COLS],
    'nulos': [df_raw[c].isna().sum() if c in df_raw.columns else len(df_raw) for c in SENSOR_COLS],
    'completitud_%': [
        round(df_raw[c].notna().mean() * 100, 1) if c in df_raw.columns else 0.0
        for c in SENSOR_COLS
    ]
})

print("📋 INVENTARIO DE VARIABLES\n")
print(completeness.to_string(index=False))

# Heatmap de nulos
available_cols = [c for c in SENSOR_COLS if c in df_raw.columns]
fig, ax = plt.subplots(figsize=(14, 2))
null_matrix = df_raw[available_cols].isna().astype(int).T
sns.heatmap(null_matrix, ax=ax, cbar=False, cmap='Reds', yticklabels=True, xticklabels=False)
ax.set_title("Mapa de valores nulos por columna (rojo = nulo)", fontweight='bold')
plt.tight_layout()
plt.show()


### 4.2 — Validación de rangos físicos

In [ ]:
PHYSICAL_RANGES = {
    'weight_grams':  (0, 5000),
    'water_ml':      (0, 2000),
    'flow_rate':     (0, 100),
    'temperature':   (-10, 60),
    'humidity':      (0, 100),
    'light_lux':     (0, 100_000),
    'light_percent': (0, 100),
    'battery_level': (0, 100),
}

print("🔬 VALIDACIÓN DE RANGOS FÍSICOS\n")
range_report = []
for col, (lo, hi) in PHYSICAL_RANGES.items():
    if col not in df_raw.columns:
        continue
    s = df_raw[col].dropna()
    out_low  = (s < lo).sum()
    out_high = (s > hi).sum()
    range_report.append({
        'variable': col,
        'min_obs': round(s.min(), 2),
        'max_obs': round(s.max(), 2),
        'rango_válido': f"[{lo}, {hi}]",
        'fuera_rango_bajo': out_low,
        'fuera_rango_alto': out_high,
        'ok': '✅' if (out_low + out_high) == 0 else '⚠️'
    })

df_range = pd.DataFrame(range_report)
print(df_range.to_string(index=False))


### 4.3 — Calidad de timestamps y clock_invalid

In [ ]:
df_ts = df_raw.copy()
df_ts['ts_diff_sec'] = (
    df_ts['ingested_at'] - df_ts['recorded_at']
).dt.total_seconds().abs()

print("⏱️  ANÁLISIS DE TIMESTAMPS\n")
print(f"  Total lecturas:          {len(df_ts):,}")
if 'clock_invalid' in df_ts.columns:
    n_invalid = df_ts['clock_invalid'].sum()
    print(f"  Clock invalid:           {n_invalid:,} ({n_invalid/len(df_ts)*100:.1f}%)")
print(f"  Desvío mediano (seg):    {df_ts['ts_diff_sec'].median():.1f}")
print(f"  Desvío p95 (seg):        {df_ts['ts_diff_sec'].quantile(0.95):.1f}")
print(f"  Desvío máximo (seg):     {df_ts['ts_diff_sec'].max():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribución de desvíos
clip_val = df_ts['ts_diff_sec'].quantile(0.99)
axes[0].hist(df_ts['ts_diff_sec'].clip(upper=clip_val), bins=60, color='steelblue', edgecolor='none')
axes[0].set_title("Distribución de desvío recorded_at vs ingested_at", fontweight='bold')
axes[0].set_xlabel("Desvío (segundos)")
axes[0].set_ylabel("Frecuencia")

# Timeline de clock_invalid
df_ts['effective_ts'] = effective_ts(df_ts)
df_ts_daily = df_ts.set_index('effective_ts').resample('D')['clock_invalid'].sum()
axes[1].bar(df_ts_daily.index, df_ts_daily.values, color='tomato', width=0.8)
axes[1].set_title("Lecturas con clock_invalid por día", fontweight='bold')
axes[1].set_xlabel("Fecha")
axes[1].set_ylabel("Conteo")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()


## 5 ? Feature engineering (Capa 2 + 3)

### 5.1 ? Derivadas por lectura (MAIN)
Se calcula `df` a partir de `df_raw`:
- `effective_ts` (elige `ingested_at` cuando `clock_invalid=True`, si no `recorded_at`)
- `dt_seconds`, `delta_food_g`, `delta_water_ml`
- `rate_g_per_min`
- transformaciones `log10(x+1)`

Reglas de seguridad:
- Nunca mezcla series entre dispositivos (todo va por `groupby(device_id)`).
- Usa `device_type` para evitar calcular comida en bowls de agua (y viceversa).
- Detecta si `readings.device_id` se une por `devices.id` o por `devices.device_id`.

> Si `readings` no trae métricas (weight/water), este bloque seguir? funcionando pero `food_content_g/water_content_cm3` quedar?n vac?os; usa `df_daily/df_sessions`.


In [ ]:
# Plate weight por device (para scope=all no puede ser un escalar)
plate_weight_map = {}
if 'plate_weight_grams' in df_devices.columns:
    plate_weight_map = (
        df_devices[['id', 'plate_weight_grams']]
        .dropna(subset=['id'])
        .set_index('id')['plate_weight_grams']
        .fillna(0)
        .to_dict()
    )

# Base
df = df_raw.copy()
df['effective_ts'] = effective_ts(df)

# Orden y derivadas por device (evita mezclar series entre dispositivos)
df = df.sort_values(['device_id', 'effective_ts']).reset_index(drop=True)

# Tara por fila
if 'weight_grams' in df.columns:
    df['plate_weight_grams'] = df['device_id'].map(plate_weight_map).fillna(0).astype(float)
    df['food_content_g'] = (df['weight_grams'] - df['plate_weight_grams']).clip(lower=0)
else:
    df['plate_weight_grams'] = np.nan
    df['food_content_g'] = np.nan

# Agua
if 'water_ml' in df.columns:
    df['water_content_cm3'] = df['water_ml']
else:
    df['water_content_cm3'] = np.nan

# Intervalos entre lecturas (por device)
df['dt_seconds'] = df.groupby('device_id')['effective_ts'].diff().dt.total_seconds()

# Delta de contenido (por device)
df['delta_food_g'] = df.groupby('device_id')['food_content_g'].diff()
df['delta_water_ml'] = df.groupby('device_id')['water_content_cm3'].diff()

# Tasa de consumo (g/min) ? deltas negativos significativos
df['rate_g_per_min'] = np.where(
    (df['delta_food_g'] < -1) & (df['dt_seconds'] > 0),
    abs(df['delta_food_g']) / (df['dt_seconds'] / 60),
    np.nan
)

# Transformaciones log10(x+1) para variables sesgadas
for col in ['weight_grams', 'water_ml', 'food_content_g', 'flow_rate', 'dt_seconds']:
    if col in df.columns:
        s = pd.to_numeric(df[col], errors='coerce')
        df[f'{col}_log'] = np.log10(s.clip(lower=0) + 1)


print("? Derivadas calculadas")
print(f"   filas: {len(df):,}")
if 'food_content_g' in df.columns:
    print(f"   food_content_g median={df['food_content_g'].median():.1f}")


### 5.2 — Baselines por ventana temporal

In [ ]:
def compute_baseline_ts(df: pd.DataFrame, ts_col: str, value_col: str, windows_days: list[int]) -> pd.DataFrame:
    # Baselines (media/mediana/std/MAD/IQR) para una serie temporal en múltiples ventanas.
    if df.empty or ts_col not in df.columns or value_col not in df.columns:
        return pd.DataFrame()

    tmp = df[[ts_col, value_col]].copy()
    tmp[ts_col] = pd.to_datetime(tmp[ts_col], utc=True, errors='coerce')
    tmp[value_col] = pd.to_numeric(tmp[value_col], errors='coerce')
    tmp = tmp.dropna(subset=[ts_col, value_col])
    if tmp.empty:
        return pd.DataFrame()

    now = tmp[ts_col].max()
    results = []
    for d in windows_days:
        cutoff = now - pd.Timedelta(days=d)
        subset = tmp[tmp[ts_col] >= cutoff][value_col].dropna()
        if len(subset) == 0:
            continue
        med = subset.median()
        results.append({
            'ventana': f'{d}d',
            'n': int(len(subset)),
            'media': round(float(subset.mean()), 2),
            'mediana': round(float(med), 2),
            'std': round(float(subset.std()), 2),
            'mad': round(float((subset - med).abs().median()), 2),
            'q25': round(float(subset.quantile(0.25)), 2),
            'q75': round(float(subset.quantile(0.75)), 2),
        })
    return pd.DataFrame(results)


def _print_baseline(title: str, b: pd.DataFrame) -> None:
    print(title)
    if b.empty:
        print('Empty DataFrame')
    else:
        print(b.to_string(index=False))


windows = [1, 7, 14, 30]

source = None
if 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty and 'effective_ts' in df.columns:
    has_food = ('food_content_g' in df.columns) and df['food_content_g'].notna().any()
    has_water = ('water_content_cm3' in df.columns) and df['water_content_cm3'].notna().any()
    if has_food or has_water:
        source = 'readings'

if source is None and 'df_daily' in globals() and isinstance(df_daily, pd.DataFrame) and not df_daily.empty:
    source = 'daily'

if source is None and 'df_sessions' in globals() and isinstance(df_sessions, pd.DataFrame) and not df_sessions.empty:
    source = 'sessions'

print('')
print(f"?? BASELINES (source={source or 'none'})")

if source == 'readings':
    bl_food = compute_baseline_ts(df, 'effective_ts', 'food_content_g', windows)
    _print_baseline('? food_content_g (readings)', bl_food)

    bl_water = compute_baseline_ts(df, 'effective_ts', 'water_content_cm3', windows)
    print('')
    _print_baseline('? water_content_cm3 (readings)', bl_water)

elif source == 'daily':
    bl_food = compute_baseline_ts(df_daily, 'summary_date', 'total_food_grams', windows)
    _print_baseline('? total_food_grams (daily)', bl_food)

    bl_water = compute_baseline_ts(df_daily, 'summary_date', 'total_water_ml', windows)
    print('')
    _print_baseline('? total_water_ml (daily)', bl_water)

elif source == 'sessions':
    bl_food = compute_baseline_ts(df_sessions, 'session_start', 'grams_consumed', windows)
    _print_baseline('? grams_consumed (sessions)', bl_food)

    bl_water = compute_baseline_ts(df_sessions, 'session_start', 'water_ml', windows)
    print('')
    _print_baseline('? water_ml (sessions)', bl_water)

else:
    print('?? No hay datos suficientes en lecturas/sesiones/resumen diario para calcular baselines en esta ventana.')
    print('   - Revisa WINDOW_DAYS / MAX_ROWS')
    print('   - Verifica si readings trae weight_grams/water_ml o si tus métricas viven sólo en analytics')


## 6 · Análisis exploratorio (EDA)

### 6.1 — Series temporales de sensores principales

In [ ]:
# Series temporales (agregado por hora)

plot_cols = [
    ('food_content_g',    'Comida (g)',            'steelblue'),
    ('water_content_cm3', 'Agua (ml)',             'mediumseagreen'),
    ('temperature',       'Temperatura (°C)',      'tomato'),
    ('humidity',          'Humedad (%)',           'mediumpurple'),
]

if 'effective_ts' not in df.columns:
    raise ValueError('Falta effective_ts (ejecuta la celda de derivadas primero)')

df_plot = df.copy()
df_plot['effective_ts'] = pd.to_datetime(df_plot['effective_ts'], utc=True, errors='coerce')
df_plot = df_plot.dropna(subset=['effective_ts']).sort_values('effective_ts')

fig, axes = plt.subplots(len(plot_cols), 1, figsize=(16, 3 * len(plot_cols)), sharex=True)
if len(plot_cols) == 1:
    axes = [axes]

scope_label = f"Pet {TARGET_PET_ID[:8]}..." if TARGET_PET_ID else 'ALL pets/devices'
fig.suptitle(f"Series temporales — {scope_label}", fontweight='bold', fontsize=13)

for ax, (col, label, color) in zip(axes, plot_cols):
    if col not in df_plot.columns:
        ax.text(0.5, 0.5, f'{col} no disponible', transform=ax.transAxes, ha='center')
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.2)
        continue

    series = pd.to_numeric(df_plot.set_index('effective_ts')[col], errors='coerce')
    ts_series = series.resample('1h').median()

    if ts_series.dropna().empty:
        ax.text(0.5, 0.5, f'{col}: sin datos', transform=ax.transAxes, ha='center')
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.2)
        continue

    ax.plot(ts_series.index, ts_series.values, color=color, linewidth=1.0, alpha=0.9)
    ax.fill_between(ts_series.index, ts_series.values, alpha=0.12, color=color)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))

plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()


### 6.2 — Distribuciones: raw vs log10

In [ ]:
log_pairs = [
    ('food_content_g', 'food_content_g_log'),
    ('water_content_cm3', 'water_ml_log'),
]

fig, axes = plt.subplots(len(log_pairs), 2, figsize=(14, 4 * len(log_pairs)))
fig.suptitle("Distribuciones: raw vs log10(x+1)", fontweight='bold')

for i, (raw_col, log_col) in enumerate(log_pairs):
    for j, (col, title) in enumerate([(raw_col, 'Raw'), (log_col, 'log10(x+1)')]):
        ax = axes[i][j] if len(log_pairs) > 1 else axes[j]
        if col not in df.columns:
            continue
        data = df[col].dropna()
        ax.hist(data, bins=60, color='steelblue' if j == 0 else 'darkorange',
                edgecolor='none', alpha=0.8)
        ax.axvline(data.median(), color='red', linestyle='--', label=f'Mediana={data.median():.2f}')
        ax.set_title(f"{raw_col} — {title}")
        ax.set_xlabel(col)
        ax.set_ylabel("Frecuencia")
        ax.legend(fontsize=9)

        # Test de normalidad
        sample = data.sample(min(len(data), 50), random_state=42)
        _, p = shapiro(sample)
        verdict = "✅ Normal" if p > 0.05 else "⚠️ No normal"
        ax.set_xlabel(f"{col}\nShapiro-Wilk p={p:.4f} → {verdict}")

plt.tight_layout()
plt.show()


### 6.3 — Correlación entre variables

In [ ]:
corr_cols = [c for c in ['food_content_g', 'water_content_cm3', 'temperature',
                          'humidity', 'light_percent', 'battery_level', 'dt_seconds']
             if c in df.columns]

corr_matrix = df[corr_cols].corr(method='spearman')  # Spearman: robusto a no-normalidad

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=~mask, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
ax.set_title("Correlación de Spearman entre variables", fontweight='bold')
plt.tight_layout()
plt.show()


## 7 ? Detección de anomalías (Capa 3)

Este bloque ilustra detectores simples y auditables:
- Z-score / IQR / MAD-score
- Tests no param?tricos (Mann?Whitney U / KS) para cambios de patr?n

Recomendaci?n:
- Para `scope=all`, interpreta estas métricas como *señales globales* (mezcla de dispositivos en una ventana).
- Para alertas por mascota/dispositivo, usa `ANALYSIS_SCOPE=pet` o agrega agrupaciones explícitas por `device_id`/`pet_id`.


In [ ]:
def test_normality(series: pd.Series, label: str) -> dict:
    """Shapiro-Wilk sobre muestra de hasta 50 obs."""
    data = series.dropna()
    sample = data.sample(min(len(data), 50), random_state=42)
    stat, p = shapiro(sample)
    is_normal = p > 0.05
    recommended = 'z-score' if is_normal else 'MAD-score / IQR'
    print(f"  {label:25s}  Shapiro p={p:.4f}  "
          f"{'✅ Normal → ' if is_normal else '⚠️ No normal → '}"
          f"usar {recommended}")
    return {'label': label, 'p_shapiro': p, 'is_normal': is_normal, 'recommended': recommended}

# Usar ventana 7d para baseline
cutoff_7d = df['effective_ts'].max() - pd.Timedelta(days=7)
df_7d = df[df['effective_ts'] >= cutoff_7d]

print("🔬 TEST DE NORMALIDAD (Shapiro-Wilk, ventana 7d)\n")
normality_results = []
for col in ['food_content_g', 'water_content_cm3', 'dt_seconds', 'temperature', 'humidity']:
    if col in df_7d.columns and df_7d[col].dropna().shape[0] >= 10:
        normality_results.append(test_normality(df_7d[col], col))


### 7.2 — Z-score, IQR y MAD-score

In [ ]:
def compute_anomaly_scores(series: pd.Series, baseline: pd.Series) -> pd.DataFrame:
    """
    Calcula z-score, IQR-score y MAD-score para cada punto en `series`
    usando estadísticos de `baseline` como referencia.
    
    Returns DataFrame con scores individuales y score combinado [0, 1].
    """
    result = pd.DataFrame({'value': series})

    # ── Z-score ────────────────────────────────────────────────────────────────
    mu, sigma = baseline.mean(), baseline.std()
    result['z_score'] = np.abs((series - mu) / sigma) if sigma > 0 else 0.0

    # ── IQR ────────────────────────────────────────────────────────────────────
    q1, q3 = baseline.quantile(0.25), baseline.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    result['iqr_score'] = np.maximum(
        (lower_fence - series) / iqr,
        (series - upper_fence) / iqr
    ).clip(lower=0)

    # ── MAD-score ──────────────────────────────────────────────────────────────
    median_bl = baseline.median()
    mad = (baseline - median_bl).abs().median()
    result['mad_score'] = np.abs(0.6745 * (series - median_bl) / mad) if mad > 0 else 0.0

    # ── Normalizar a [0, 1] y combinar ────────────────────────────────────────
    for col in ['z_score', 'iqr_score', 'mad_score']:
        mx = result[col].max()
        result[f'{col}_norm'] = result[col] / mx if mx > 0 else 0.0

    result['anomaly_score'] = (
        0.3 * result['z_score_norm'] +
        0.4 * result['iqr_score_norm'] +
        0.3 * result['mad_score_norm']
    )

    def classify(score):
        if score >= 0.7: return 'alert'
        if score >= 0.4: return 'warning'
        return 'normal'

    result['severity'] = result['anomaly_score'].apply(classify)
    return result


# Aplicar sobre food_content_g: baseline=7d, detección en últimas 24h
cutoff_24h = df['effective_ts'].max() - pd.Timedelta(hours=24)
baseline_food = df_7d['food_content_g'].dropna()
recent_food   = df[df['effective_ts'] >= cutoff_24h]['food_content_g'].dropna()

if len(baseline_food) > 5 and len(recent_food) > 0:
    scores_food = compute_anomaly_scores(recent_food, baseline_food)

    print("📊 ANOMALY SCORES — food_content_g (últimas 24h)\n")
    print(scores_food[['value','z_score','iqr_score','mad_score','anomaly_score','severity']]
          .describe().round(3))
    print(f"\n  Alertas:   {(scores_food['severity']=='alert').sum()}")
    print(f"  Warnings:  {(scores_food['severity']=='warning').sum()}")
    print(f"  Normales:  {(scores_food['severity']=='normal').sum()}")
else:
    print("⚠️ Datos insuficientes para calcular anomaly scores")


### 7.3 — Visualización de anomalías sobre la serie

In [ ]:
if len(baseline_food) > 5 and len(recent_food) > 0:
    # Reconstruir timestamps para scores_food
    df_recent = df[df['effective_ts'] >= cutoff_24h][['effective_ts', 'food_content_g']].dropna().copy()
    df_recent = df_recent.reset_index(drop=True)
    scores_food = scores_food.reset_index(drop=True)
    df_recent['anomaly_score'] = scores_food['anomaly_score'].values[:len(df_recent)]
    df_recent['severity']      = scores_food['severity'].values[:len(df_recent)]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
    fig.suptitle("Detección de anomalías — food_content_g (24h)", fontweight='bold')

    # Serie + puntos coloreados por severidad
    color_map = {'normal': 'steelblue', 'warning': 'orange', 'alert': 'red'}
    ax1.plot(df_recent['effective_ts'], df_recent['food_content_g'],
             color='lightgray', linewidth=1, zorder=1)
    for sev, color in color_map.items():
        mask = df_recent['severity'] == sev
        ax1.scatter(df_recent.loc[mask, 'effective_ts'],
                    df_recent.loc[mask, 'food_content_g'],
                    c=color, s=30, label=sev, zorder=2, alpha=0.8)

    # Banda de IQR del baseline
    q1, q3 = baseline_food.quantile(0.25), baseline_food.quantile(0.75)
    iqr = q3 - q1
    ax1.axhspan(q1 - 1.5 * iqr, q3 + 1.5 * iqr, alpha=0.08, color='green', label='IQR baseline')
    ax1.axhline(baseline_food.median(), color='green', linestyle='--', alpha=0.5, label='Mediana baseline')
    ax1.set_ylabel("food_content_g")
    ax1.legend(fontsize=9)

    # Anomaly score timeline
    ax2.fill_between(df_recent['effective_ts'], df_recent['anomaly_score'],
                     alpha=0.4, color='tomato')
    ax2.plot(df_recent['effective_ts'], df_recent['anomaly_score'], color='tomato', linewidth=1)
    ax2.axhline(0.7, color='red',    linestyle='--', alpha=0.7, label='Umbral alert (0.7)')
    ax2.axhline(0.4, color='orange', linestyle='--', alpha=0.7, label='Umbral warning (0.4)')
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("anomaly_score")
    ax2.set_xlabel("Tiempo")
    ax2.legend(fontsize=9)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

    plt.tight_layout()
    plt.show()


### 7.4 — Test de cambio de patrón (Mann-Whitney U + KS)

In [ ]:
def test_pattern_change(recent: pd.Series, baseline: pd.Series, label: str):
    """
    Mann-Whitney U + KS test para detectar cambio de distribución.
    H₀: las dos muestras provienen de la misma distribución.
    """
    if len(recent) < 5 or len(baseline) < 5:
        print(f"  {label}: datos insuficientes")
        return

    mw_stat, mw_p = mannwhitneyu(recent, baseline, alternative='two-sided')
    ks_stat, ks_p = ks_2samp(recent, baseline)

    mw_changed = mw_p < 0.05
    ks_changed = ks_p < 0.05
    conclusion = "🔴 CAMBIO detectado" if (mw_changed and ks_changed) else \
                 "🟡 Posible cambio"   if (mw_changed or ks_changed) else \
                 "🟢 Sin cambio"

    print(f"  {label}")
    print(f"    Mann-Whitney U: stat={mw_stat:.1f}  p={mw_p:.4f}  "
          f"{'Rechaza H₀ ✓' if mw_changed else 'No rechaza H₀'}")
    print(f"    KS:             stat={ks_stat:.3f}   p={ks_p:.4f}  "
          f"{'Rechaza H₀ ✓' if ks_changed else 'No rechaza H₀'}")
    print(f"    Conclusión: {conclusion}")
    print(f"    Baseline median={baseline.median():.2f}  Recent median={recent.median():.2f}")
    print()

    # Visualización de distribuciones superpuestas
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"Cambio de distribución — {label}", fontweight='bold')

    ax1.hist(baseline, bins=30, alpha=0.6, color='steelblue', label='Baseline 7d', density=True)
    ax1.hist(recent,   bins=30, alpha=0.6, color='tomato',    label='Reciente 3d',  density=True)
    ax1.axvline(baseline.median(), color='steelblue', linestyle='--', linewidth=1.5)
    ax1.axvline(recent.median(),   color='tomato',    linestyle='--', linewidth=1.5)
    ax1.set_title("Densidades superpuestas")
    ax1.legend()

    ax2.ecdf(baseline, label='Baseline 7d', color='steelblue')
    ax2.ecdf(recent,   label='Reciente 3d',  color='tomato')
    ax2.set_title(f"CDFs acumuladas (KS stat={ks_stat:.3f})")
    ax2.set_ylabel("Probabilidad acumulada")
    ax2.legend()

    plt.tight_layout()
    plt.show()


cutoff_3d  = df['effective_ts'].max() - pd.Timedelta(days=3)
recent_3d  = df[df['effective_ts'] >= cutoff_3d]['food_content_g'].dropna()
baseline_7d = df[(df['effective_ts'] >= cutoff_7d) &
                 (df['effective_ts'] < cutoff_3d)]['food_content_g'].dropna()

print("🧪 TEST DE CAMBIO DE PATRÓN\n")
test_pattern_change(recent_3d, baseline_7d, 'food_content_g')


## 8 · Análisis de rutinas (FFT / Fourier)

In [ ]:
# FFT / Rutina
# Nota: FFT tiene sentido por mascota (series coherentes). Para scope=all, se omite.

fft_result = {'dominant_frequency': None, 'frequency_power': None, 'period_hours': None}

if ANALYSIS_SCOPE != 'pet':
    print("?  FFT se ejecuta solo con ANALYSIS_SCOPE=pet (series por mascota)")
else:
    def compute_fft_rutina(df: pd.DataFrame, bucket_minutes: int = 30, window_days: int = 7) -> dict:
        """
        An?lisis de Fourier sobre la serie de eventos de sesi?n.
        Detecta el per?odo dominante de alimentaci?n/hidrataci?n.
        """
        ts_col = 'effective_ts'
        cutoff = df[ts_col].max() - pd.Timedelta(days=window_days)
        df_w = df[df[ts_col] >= cutoff].copy()

        df_w['bucket'] = df_w[ts_col].dt.floor(f'{bucket_minutes}T')
        signal_series = df_w.groupby('bucket')['delta_food_g'].sum()

        full_idx = pd.date_range(
            start=df_w[ts_col].min().floor(f'{bucket_minutes}T'),
            end=df_w[ts_col].max().ceil(f'{bucket_minutes}T'),
            freq=f'{bucket_minutes}T'
        )
        signal = signal_series.reindex(full_idx, fill_value=0).fillna(0).values
        signal_binary = (np.abs(signal) > signal.std()).astype(float)

        n = len(signal_binary)
        fft_vals = np.fft.rfft(signal_binary)
        magnitudes = np.abs(fft_vals)
        freqs = np.fft.rfftfreq(n, d=bucket_minutes / 60.0)  # ciclos/hora

        magnitudes[0] = 0  # ignorar DC
        idx = int(np.argmax(magnitudes))
        dominant_freq = float(freqs[idx]) if idx < len(freqs) else 0.0

        period_hours = (1 / dominant_freq) if dominant_freq > 0 else None
        power = float(magnitudes[idx] / (magnitudes.max() + 1e-9)) if len(magnitudes) else 0.0

        return {
            'dominant_frequency': round(dominant_freq, 4),
            'frequency_power': round(power, 4),
            'period_hours': round(period_hours, 2) if period_hours else None
        }

    fft_result = compute_fft_rutina(df)
    print("? FFT rutina:", fft_result)


## 9 · Resumen diario y tendencias

In [ ]:
if not df_daily.empty:
    fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
    fig.suptitle("Resumen diario de la mascota", fontweight='bold')

    # Comida y agua
    ax = axes[0]
    if 'total_food_grams' in df_daily.columns:
        ax.bar(df_daily['summary_date'], df_daily['total_food_grams'],
               color='steelblue', alpha=0.7, label='Comida (g)')
    ax2_twin = ax.twinx()
    if 'total_water_ml' in df_daily.columns:
        ax2_twin.plot(df_daily['summary_date'], df_daily['total_water_ml'],
                      color='mediumseagreen', marker='o', markersize=3, label='Agua (ml)')
    ax.set_ylabel("Comida (g)"); ax2_twin.set_ylabel("Agua (ml)")
    ax.set_title("Consumo diario")
    ax.legend(loc='upper left'); ax2_twin.legend(loc='upper right')

    # Sesiones y anomalías
    ax = axes[1]
    if 'food_sessions' in df_daily.columns:
        ax.bar(df_daily['summary_date'], df_daily['food_sessions'],
               color='steelblue', alpha=0.5, label='Sesiones comida')
    if 'anomaly_count' in df_daily.columns:
        ax_twin = ax.twinx()
        ax_twin.plot(df_daily['summary_date'], df_daily['anomaly_count'],
                     color='tomato', marker='x', label='Anomalías')
        ax_twin.set_ylabel("Anomalías")
        ax_twin.legend(loc='upper right')
    ax.set_ylabel("Sesiones")
    ax.set_title("Sesiones y anomalías diarias")
    ax.legend(loc='upper left')

    # Temperatura y humedad ambiente
    ax = axes[2]
    if 'avg_temperature' in df_daily.columns:
        ax.plot(df_daily['summary_date'], df_daily['avg_temperature'],
                color='tomato', marker='o', markersize=3, label='Temperatura (°C)')
    if 'avg_humidity' in df_daily.columns:
        ax_twin = ax.twinx()
        ax_twin.plot(df_daily['summary_date'], df_daily['avg_humidity'],
                     color='mediumpurple', marker='s', markersize=3, label='Humedad (%)')
        ax_twin.set_ylabel("Humedad (%)")
        ax_twin.legend(loc='upper right')
    ax.set_ylabel("Temperatura (°C)")
    ax.set_title("Ambiente diario")
    ax.legend(loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ pet_daily_summary no disponible — saltando esta sección")


## 10 · Reporte de insights automáticos

In [ ]:
from datetime import datetime, timezone

insights = []
now = df['effective_ts'].max()

# ── Baseline 7d ────────────────────────────────────────────────────────────────
food_24h   = df[df['effective_ts'] >= now - pd.Timedelta(hours=24)]['food_content_g'].sum()
water_24h  = df[df['effective_ts'] >= now - pd.Timedelta(hours=24)]['water_content_cm3'].sum()
food_bl7d  = df_7d['food_content_g'].sum() / 7
water_bl7d = df_7d['water_content_cm3'].sum() / 7

# ── Insight: hidratación baja ──────────────────────────────────────────────────
if water_bl7d > 0 and water_24h < 0.7 * water_bl7d:
    pct = round((1 - water_24h / water_bl7d) * 100, 1)
    insights.append({
        'type': 'hydration_low',
        'severity': 'warning',
        'message': f'Bebió {pct}% menos agua que su promedio de 7 días.',
        'context': {'actual_ml': round(water_24h,1), 'baseline_ml': round(water_bl7d,1)}
    })

# ── Insight: ayuno prolongado ──────────────────────────────────────────────────
last_food_event = df[df['delta_food_g'] < -2]['effective_ts'].max()
if pd.notna(last_food_event):
    hours_since = (now - last_food_event).total_seconds() / 3600
    if hours_since > 12:
        insights.append({
            'type': 'fasting_prolonged',
            'severity': 'alert',
            'message': f'Sin ingesta de comida por {hours_since:.1f} horas.',
            'context': {'hours_since_last_meal': round(hours_since, 1)}
        })

# ── Insight: rutina (FFT) ──────────────────────────────────────────────────────
if fft_result['frequency_power'] < 0.4:
    insights.append({
        'type': 'irregular_routine',
        'severity': 'info',
        'message': 'Rutina de alimentación irregular en los últimos 7 días.',
        'context': {'frequency_power': fft_result['frequency_power'],
                    'period_hours': fft_result['period_hours']}
    })

# ── Insight: cambio de patrón (Mann-Whitney) ──────────────────────────────────
if len(recent_3d) >= 5 and len(baseline_7d) >= 5:
    _, mw_p = mannwhitneyu(recent_3d, baseline_7d, alternative='two-sided')
    if mw_p < 0.05:
        insights.append({
            'type': 'pattern_change',
            'severity': 'alert',
            'message': 'Cambio estadísticamente significativo en el patrón de consumo.',
            'context': {'mannwhitney_p': round(mw_p, 4),
                        'baseline_median_g': round(baseline_7d.median(), 2),
                        'recent_median_g':   round(recent_3d.median(), 2)}
        })

# ── Print reporte ──────────────────────────────────────────────────────────────
SEVERITY_EMOJI = {'info': 'ℹ️', 'warning': '⚠️', 'alert': '🔴'}
print("=" * 60)
print(f"  REPORTE DE INSIGHTS — {now.strftime('%Y-%m-%d %H:%M')} UTC")
print("=" * 60)

if not insights:
    print("\n  ✅ Sin anomalías ni alertas detectadas.\n")
else:
    for ins in insights:
        emoji = SEVERITY_EMOJI.get(ins['severity'], '•')
        print(f"\n  {emoji} [{ins['severity'].upper()}] {ins['type']}")
        print(f"     {ins['message']}")
        print(f"     Contexto: {ins['context']}")

print("\n" + "=" * 60)


## 11 ? Validación de modelos: métricas de calidad

Objetivo: validar estabilidad y drift sin asumir un modelo complejo.

Incluye (según datos disponibles):
- Res?menes de distribuci?n
- PSI (Population Stability Index) para drift entre baseline vs periodo reciente
- Checks de ?datos insuficientes? para evitar conclusiones falsas


In [ ]:
def compute_psi(expected: pd.Series, actual: pd.Series, n_bins: int = 10) -> float:
    """
    Population Stability Index (PSI).
    Detecta data drift entre baseline (expected) y datos actuales (actual).
    PSI < 0.10 → sin drift
    PSI 0.10–0.25 → monitorear
    PSI > 0.25 → reentrenar modelo
    """
    breakpoints = np.percentile(expected.dropna(), np.linspace(0, 100, n_bins + 1))
    breakpoints  = np.unique(breakpoints)

    def bucket_pcts(s):
        counts, _ = np.histogram(s.dropna(), bins=breakpoints)
        pcts = counts / len(s.dropna())
        return np.clip(pcts, 1e-4, None)  # evitar log(0)

    exp_pcts = bucket_pcts(expected)
    act_pcts = bucket_pcts(actual)
    min_len  = min(len(exp_pcts), len(act_pcts))
    psi = np.sum((act_pcts[:min_len] - exp_pcts[:min_len]) *
                 np.log(act_pcts[:min_len] / exp_pcts[:min_len]))
    return round(float(psi), 4)

print("📏 MÉTRICAS DE VALIDACIÓN DE MODELOS\n")
print("  Population Stability Index (PSI) — data drift:")
print("  " + "-" * 50)

feature_pairs = [
    ('food_content_g',    df_7d['food_content_g'].dropna(),   recent_3d),
    ('water_content_cm3', df_7d['water_content_cm3'].dropna(),
     df[df['effective_ts'] >= cutoff_3d]['water_content_cm3'].dropna()),
]

for label, exp, act in feature_pairs:
    if len(exp) < 10 or len(act) < 5:
        print(f"  {label:25s}  datos insuficientes")
        continue
    psi = compute_psi(exp, act)
    status = ('✅ Sin drift' if psi < 0.10
              else '🟡 Monitorear' if psi < 0.25
              else '🔴 Reentrenar')
    print(f"  {label:25s}  PSI={psi:.4f}  {status}")

print()
print("  Anomaly score — distribución en últimas 24h:")
print("  " + "-" * 50)
if 'scores_food' in dir() and len(scores_food) > 0:
    score_summary = scores_food['anomaly_score'].describe().round(3)
    for stat, val in score_summary.items():
        print(f"  {stat:8s}: {val}")
    severity_dist = scores_food['severity'].value_counts(normalize=True).round(3)
    print("\n  Distribución de severidad:")
    for sev, pct in severity_dist.items():
        print(f"  {sev:10s}: {pct*100:.1f}%")


---
## Próximos pasos

| Prioridad | Tarea | Sección de referencia |
|---|---|---|
| Alta | Etiquetar manualmente sesiones de ground truth para calcular Precision/Recall | Sec. 11 |
| Alta | Persistir `anomaly_score` en `public.pet_sessions` desde el worker | Doc. arquitectura Capa 3 |
| Media | Ajustar umbrales de anomalía (0.4 / 0.7) con datos reales | Sec. 7.2 |
| Media | Crear tabla `public.insights_ml` y migrar reporte de Sec. 10 al worker | Doc. arquitectura Sec. 6 |
| Baja | Extender FFT a múltiples mascotas y comparar rutinas | Sec. 8 |
| Baja | Agregar modelo ARIMA/Prophet para predicción de consumo futuro | — |